# 📦 The LangChain Package Split — where your imports went

## Learning Objectives
In this notebook, you will learn:
1. **Why `langchain` shrank** - the reasoning behind 1.0's package reorganization
2. **The four destinations** - `langchain`, `langchain-core`, partner packages, `langchain-classic`
3. **Reading the error** - turning an `ImportError` into the exact line you need
4. **The 1.x re-exports** - the short import paths `langchain` still offers

## Prerequisites
- `langchain >= 1.2.7` and `langchain-openai` installed
- `OPENAI_API_KEY` in your project `.env` (only used by the last code cell)
- Notebook `1.0_LangChain_Introduction`

---
## 🤔 Part 1: Why this changed

Through 0.x, the `langchain` package was the whole ecosystem: chains, agents, memory,
retrievers, every provider integration, the prompt hub. Installing it pulled in a large
surface, and *any* integration breaking could break the import.

1.0 split that surface by **rate of change, and by who owns each piece**:

| Package | Holds | Changes |
| --- | --- | --- |
| `langchain-core` | base abstractions — messages, documents, prompts, parsers, `Runnable` | rarely, deliberately |
| `langchain` | the agent loop + convenience re-exports | with the agent API |
| `langchain-openai`, `langchain-groq`, … | one provider each | at that provider's pace |
| `langchain-classic` | everything retired: chains, retrievers, indexing, hub | frozen |

The practical consequence is the one you feel: **a 0.x import path is not "deprecated", it is
gone.** There is no shim and no warning — the module does not exist.

### Key Concepts:
- **`langchain-core`**: the stable base every other package builds on
- **`langchain-classic`**: a real, installable package — legacy code still runs, it just moved.
  "Moved" is not the same as "recommended": some of what lives there (memory, most chains) has
  a better 1.x answer than importing the old class.

---
## ⚙️ Part 2: Setup

This track instantiates providers directly rather than through the repo's `helpers` factory,
matching the other notebooks in `LangChain_Fundamentals/`.

In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP: load credentials and report installed versions
# ==============================================================================
import importlib.metadata as md

from dotenv import load_dotenv

load_dotenv()

for pkg in ("langchain", "langchain-core", "langchain-openai"):
    try:
        print(f"📋 {pkg:<20} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"⚠️  {pkg:<20} not installed")

print("✅ Setup complete!")

---
## 🕰️ Part 3: The old imports

> ⚠️ Does not run on LangChain 1.x — shown for contrast only.

Every line below worked in 0.x. Every one fails now — but **not with the same error**.
The five module paths raise `ModuleNotFoundError`; `from langchain import hub` raises
`ImportError`, because `langchain` still exists as a package, it just no longer exports
that name. Knowing which of the two you got tells you whether a *module* moved or a
*name* did.

In [ ]:
# ==============================================================================
# LEGACY 0.X: IMPORT PATHS THAT NO LONGER EXIST
# ==============================================================================
from langchain.chains import LLMChain, RetrievalQA
from langchain.schema import Document, HumanMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.memory import ConversationBufferMemory
from langchain.llms import OpenAI
from langchain import hub

---
## 🔍 Part 4: Proving it, and finding the new home

Rather than trust a table, ask the interpreter. The cell below tries each old path and reports
what actually happens, then imports the 1.x replacements to show they resolve.

In [ ]:
# ==============================================================================
# IMPORT AUDIT: which 0.x paths are gone, and what replaced them
# ==============================================================================
import importlib

MOVED = {
    "langchain.chains": "langchain_classic.chains",
    "langchain.schema": "langchain_core.messages / .documents / .output_parsers",
    "langchain.text_splitter": "langchain_text_splitters",
    "langchain.memory": "langchain_classic.memory (legacy) / a LangGraph checkpointer",
    "langchain.llms": "langchain_openai / langchain_groq / ...",
    "langchain.retrievers": "langchain_classic.retrievers",
    "langchain.indexes": "langchain_classic.indexes",
}

for old, new in MOVED.items():
    try:
        importlib.import_module(old)
        print(f"⚠️  {old:<28} still imports (unexpected on 1.x)")
    except ModuleNotFoundError:
        print(f"❌ {old:<28} gone  ->  {new}")

# --- the replacements, which ship with any 1.x install ---
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

print()
print("✅ langchain-core replacements imported cleanly")

---
## ✨ Part 5: The 1.x re-exports

`langchain` itself keeps a small, curated surface — convenience aliases for the things you
reach for constantly. They are shorter than the `langchain_core` paths and equivalent to them.

In [ ]:
# ==============================================================================
# CONVENIENCE RE-EXPORTS: the short paths langchain 1.x still offers
# ==============================================================================
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage as HM

# `langchain.messages.HumanMessage` IS `langchain_core.messages.HumanMessage` --
# an alias, not a copy. Use whichever reads better in your file.
print(f"📋 same class object: {HM is HumanMessage}")

# init_chat_model resolves a "provider:model" string, so you can swap models
# without swapping import lines.
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)
# llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0)
# llm = init_chat_model("anthropic:claude-sonnet-4-5", temperature=0)
# Swapping provider means editing the string, not the import line -- which is
# the whole point of the re-export.

print(f"🤖 Model loaded: {llm.__class__.__name__}")
print("✅ Re-exports verified!")

---
## 🔀 Part 6: The full mapping

| 0.x import | 1.x replacement |
| --- | --- |
| `langchain.chains` | `langchain_classic.chains` — or rewrite as LCEL |
| `langchain.retrievers` | `langchain_classic.retrievers` |
| `langchain.indexes` | `langchain_classic.indexes` |
| `from langchain import hub` | `from langchain_classic import hub` |
| `langchain.schema` (messages) | `langchain_core.messages` |
| `langchain.schema` (Document) | `langchain_core.documents` |
| `langchain.schema.output_parser` | `langchain_core.output_parsers` |
| `langchain.text_splitter` | `langchain_text_splitters` |
| `langchain.document_loaders` | `langchain_community.document_loaders` |
| `langchain.vectorstores` | `langchain_chroma`, `langchain_community.vectorstores`, … |
| `langchain.llms` / `langchain.chat_models.X` | `langchain_openai`, `langchain_groq`, … |
| `langchain.embeddings.X` | the matching partner package |
| `langchain.memory` | `langchain_classic.memory` still has the classes, **but** the recommended replacement is a LangGraph checkpointer + `thread_id` |
| `langchain.globals` | `langchain_core.globals` |

---
## 🚨 Part 7: Common errors when migrating

**1. The one you will actually see**

```
ModuleNotFoundError: No module named 'langchain.chains'
```

The module is gone, not renamed in place. Install `langchain-classic` and import from there,
or rewrite the chain as LCEL.

**2. Installed but still failing**

```
ModuleNotFoundError: No module named 'langchain_classic'
```

A different problem: the *package* is not in your environment. Pinning it in
`requirements.txt` is not the same as installing it —

```bash
uv pip install langchain-classic
```

**3. A different failure for `hub`**

```
ImportError: cannot import name 'hub' from 'langchain'
```

Note the exception class: `ImportError`, not `ModuleNotFoundError`. The `langchain` package is
still importable — it simply no longer exports `hub`. Use `from langchain_classic import hub`.

**4. The alias check**

`langchain.messages.HumanMessage is langchain_core.messages.HumanMessage` returns `True`.
If you ever see `False`, you have two LangChain installs on `sys.path`.

---
## 🧪 Part 8: Try it yourself

1. Open any notebook in `04_Chains/` and, without running it, list every import that will fail
   on 1.x and write its replacement beside it. Check yourself with the audit cell in Part 4.
2. `langchain_classic.memory.ConversationBufferMemory` still exists and still imports. Yet the
   mapping table steers you to a LangGraph checkpointer instead. Re-read Part 1 and argue both
   sides: when is reaching for the classic class the right call, and when is it storing up work?

---
## 📝 Summary

### 1. The split
- **Key point**: `langchain` 1.x holds the agent loop plus re-exports; everything else moved
- **Key point**: `langchain-core` is the stable base — messages, documents, prompts, parsers

### 2. Where things went
- **Key point**: retired APIs (chains, retrievers, indexing, hub) live in `langchain-classic`
- **Key point**: provider classes live in their own partner packages
- **Key point**: `langchain.memory` moved to `langchain_classic.memory` like the rest, but it
  is the one case where the *recommended* answer is a replacement, not the moved class

### 3. Reading failures
- **Key point**: `No module named 'langchain.chains'` means the path moved
- **Key point**: `No module named 'langchain_classic'` means the package is not installed

### Next Steps
- Work through `3.5_Chain_Migrations.ipynb` for the before/after of chains → LCEL
- Then, once it lands, `8.2_Doc_Chains_to_LCEL_LangChain_v1.ipynb` for the
  document-combining chains (`load_summarize_chain`, map-reduce, refine)